# UTTA-Med Gate C — Tent + EATA (same Colab T4 session)

Run this **only after** Gate B-FULL wrote `camelyon17_resnet18_source_s42.pt`.
Keep the runtime that already cached HuggingFace Camelyon17.

Protocol (frozen):
- Continual TTA, one pass, no shuffle, `steps=1`
- Update **BatchNorm affine only** (γ, β) for both Tent and EATA
- Target labels used only for the final metrics table
- Do not tune lr / e_margin on test. If you sweep, use OOD val (center 1) only.

In [ ]:
import json, copy, time
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, average_precision_score, f1_score, roc_auc_score, precision_score, recall_score
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

assert torch.cuda.is_available(), 'Need the same T4 runtime'
device = torch.device('cuda')
print(torch.cuda.get_device_name(0))

SAVE_DIR = Path('/content/drive/MyDrive/UTTA-Med/runs')
if not SAVE_DIR.exists():
    SAVE_DIR = Path('/content/utta_med_runs')
CKPT = SAVE_DIR / 'camelyon17_resnet18_source_s42.pt'
assert CKPT.exists(), f'Missing source checkpoint {CKPT}. Finish Gate B-FULL first.'
print('ckpt', CKPT, 'MB', round(CKPT.stat().st_size/1e6, 1))

In [ ]:
# Reuse objects from notebook 01 if this kernel is still alive.
# Otherwise this cell rebuilds the eval loader from the HF cache (fast, no re-download).
need = ('model' not in globals()) or ('test_loader' not in globals()) or ('test_set' not in globals())
if need:
    print('Kernel was reset — rebuilding model + test loader from HF cache')
    from PIL import Image
    from torch.utils.data import Dataset
    from torchvision import models, transforms
    from datasets import load_dataset

    MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
    eval_tf = transforms.Compose([transforms.Resize((96,96)), transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

    def _cid(v):
        return int(v.item() if hasattr(v, 'item') else v)

    class HFCamelyon(Dataset):
        def __init__(self, hf_split, centers, transform):
            self.transform = transform
            centers = set(int(c) for c in centers)
            keep = [i for i, c in enumerate(hf_split['center']) if _cid(c) in centers]
            self.ds = hf_split.select(keep) if len(keep) < len(hf_split) else hf_split
        def __len__(self): return len(self.ds)
        def __getitem__(self, i):
            row = self.ds[i]
            img = row['image']
            if not isinstance(img, Image.Image):
                img = Image.fromarray(np.array(img)).convert('RGB')
            else:
                img = img.convert('RGB')
            y = float(int(row['label']) if not isinstance(row['label'], (int, float)) else row['label'])
            return self.transform(img), torch.tensor(y, dtype=torch.float32)

    class ResNet18Classifier(nn.Module):
        def __init__(self, dropout_p=0.5):
            super().__init__()
            bb = models.resnet18(weights=None)
            dim = bb.fc.in_features; bb.fc = nn.Identity()
            self.backbone = bb
            self.classifier = nn.Sequential(nn.Linear(dim, dim), nn.ReLU(inplace=True), nn.Dropout(dropout_p), nn.Linear(dim, 1))
        def forward(self, x): return self.classifier(self.backbone(x))

    raw = load_dataset('wltjr1007/Camelyon17-WILDS')
    test_set = HFCamelyon(raw['test'], {2}, eval_tf)
    ood_set = HFCamelyon(raw['validation'], {1}, eval_tf)
    test_loader = DataLoader(test_set, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)
    ood_loader = DataLoader(ood_set, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)
    model = ResNet18Classifier().to(device)
else:
    print('Reusing kernel objects from notebook 01')

state = torch.load(CKPT, map_location='cpu', weights_only=False)
print('loaded epoch', state.get('epoch'), 'val_auroc', state.get('val_auroc'))

In [ ]:
def ece_score(y, p, n_bins=15):
    y, p = np.asarray(y).ravel(), np.asarray(p).ravel()
    bins = np.linspace(0, 1, n_bins + 1); e = 0.0
    for i in range(n_bins):
        m = (p > bins[i]) & (p <= bins[i+1]) if i else (p >= bins[i]) & (p <= bins[i+1])
        if m.any(): e += m.mean() * abs(y[m].mean() - p[m].mean())
    return float(e)

def metrics_of(y, p):
    y, p = np.asarray(y).ravel(), np.asarray(p).ravel()
    pred = (p >= 0.5).astype(int)
    return {
        'n': int(len(y)),
        'accuracy': float(accuracy_score(y, pred)),
        'precision': float(precision_score(y, pred, zero_division=0)),
        'recall': float(recall_score(y, pred, zero_division=0)),
        'sensitivity': float(recall_score(y, pred, zero_division=0)),
        'specificity': float(((y==0)&(pred==0)).sum() / max((y==0).sum(), 1)),
        'f1': float(f1_score(y, pred, zero_division=0)),
        'auroc': float(roc_auc_score(y, p)),
        'auprc': float(average_precision_score(y, p)),
        'ece': ece_score(y, p),
        'brier': float(np.mean((p-y)**2)),
    }

def flip_table(y, before, after):
    y = np.asarray(y).ravel().astype(int)
    b = (np.asarray(before).ravel() >= 0.5).astype(int)
    a = (np.asarray(after).ravel() >= 0.5).astype(int)
    correct_b, wrong_b = (b==y), (b!=y)
    return {
        'stable': int(((b==y)&(a==y)).sum()),
        'correction': int(((b!=y)&(a==y)).sum()),
        'persistent': int(((b!=y)&(a!=y)).sum()),
        'harmful': int(((b==y)&(a!=y)).sum()),
        'correction_rate': float(((b!=y)&(a==y)).sum() / max(wrong_b.sum(), 1)),
        'harm_rate': float(((b==y)&(a!=y)).sum() / max(correct_b.sum(), 1)),
        'flip_rate': float((a!=b).mean()),
    }

def collect_bn_affine(m):
    for p in m.parameters(): p.requires_grad = False
    params = []
    for mod in m.modules():
        if isinstance(mod, nn.BatchNorm2d) and mod.affine:
            mod.weight.requires_grad = True; mod.bias.requires_grad = True
            params += [mod.weight, mod.bias]
    assert params, 'no BN affine'
    return params

def tta_mode(m):
    m.train()
    for mod in m.modules():
        if isinstance(mod, nn.Dropout): mod.eval()
        elif isinstance(mod, nn.BatchNorm2d): mod.train()

def binary_entropy(logits):
    p = torch.sigmoid(logits).clamp(1e-6, 1-1e-6)
    return -(p*torch.log(p)+(1-p)*torch.log(1-p)).view(-1)

def load_fresh():
    m = copy.deepcopy(model) if 'ResNet18Classifier' not in str(type(model)) else type(model)()
    # always rebuild from class in this kernel
    from torchvision import models as tv
    class R(nn.Module):
        def __init__(self, dropout_p=0.5):
            super().__init__()
            bb = tv.models.resnet18(weights=None)
            dim = bb.fc.in_features; bb.fc = nn.Identity(); self.backbone = bb
            self.classifier = nn.Sequential(nn.Linear(dim, dim), nn.ReLU(inplace=True), nn.Dropout(dropout_p), nn.Linear(dim, 1))
        def forward(self, x): return self.classifier(self.backbone(x))
    m = R().to(device)
    m.load_state_dict(state['model'])
    return m

In [ ]:
@torch.no_grad()
def source_only(loader):
    m = load_fresh(); m.eval()
    ys, ps = [], []
    for x, y in loader:
        x = x.to(device, non_blocking=True)
        ps.append(torch.sigmoid(m(x)).view(-1).cpu().numpy())
        ys.append(y.view(-1).numpy())
    y, p = np.concatenate(ys), np.concatenate(ps)
    return y, p, metrics_of(y, p)

def adapt_stream(loader, method, lr=1e-3, e_margin=0.4):
    m = load_fresh()
    params = collect_bn_affine(m)
    opt = torch.optim.Adam(params, lr=lr)
    n_seen = n_kept = n_skip_batches = 0
    ys, before, after = [], [], []
    src = load_fresh(); src.eval()
    t0 = time.time()
    for x, y in tqdm(loader, desc=method):
        x = x.to(device, non_blocking=True)
        with torch.no_grad():
            pb = torch.sigmoid(src(x)).view(-1).cpu().numpy()
        tta_mode(m)
        n_seen += int(x.size(0))
        if method == 'tent':
            opt.zero_grad(set_to_none=True)
            loss = binary_entropy(m(x)).mean()
            loss.backward(); opt.step()
            n_kept += int(x.size(0))
        elif method == 'eata':
            with torch.no_grad():
                keep = binary_entropy(m(x)) < e_margin
            k = int(keep.sum().item())
            if k == 0:
                n_skip_batches += 1
            else:
                opt.zero_grad(set_to_none=True)
                ent = binary_entropy(m(x))
                loss = ent[keep].mean()
                loss.backward(); opt.step()
                n_kept += k
        else:
            raise ValueError(method)
        with torch.no_grad():
            m.eval()
            pa = torch.sigmoid(m(x)).view(-1).cpu().numpy()
        before.append(pb); after.append(pa); ys.append(y.view(-1).numpy())
    y = np.concatenate(ys); pb = np.concatenate(before); pa = np.concatenate(after)
    return {
        'method': method, 'lr': lr, 'e_margin': e_margin if method=='eata' else None,
        'protocol': 'continual_one_pass_no_shuffle_bn_affine',
        'seconds': round(time.time()-t0, 1),
        'n_seen': n_seen, 'n_adapted': n_kept,
        'coverage': n_kept / max(n_seen, 1),
        'n_skip_batches': n_skip_batches,
        'before': metrics_of(y, pb),
        'after': metrics_of(y, pa),
        'harm': flip_table(y, pb, pa),
    }

print('source-only target (eval only)...')
y_t, p_t, src_test = source_only(test_loader)
print('source test AUROC', round(src_test['auroc'], 4), 'F1', round(src_test['f1'], 4), 'ECE', round(src_test['ece'], 4))

print('\nTent on target...')
tent = adapt_stream(test_loader, 'tent', lr=1e-3)
print('EATA on target...')
eata = adapt_stream(test_loader, 'eata', lr=1e-3, e_margin=0.4)

out = {
    'cuda': True,
    'device': torch.cuda.get_device_name(0),
    'checkpoint': str(CKPT),
    'source_test': src_test,
    'tent_test': tent,
    'eata_test': eata,
}
path = SAVE_DIR / 'camelyon17_resnet18_tent_eata_s42.json'
path.write_text(json.dumps(out, indent=2))
print('\n=== GATE C ===')
for name, r in [('source', {'after': src_test, 'coverage': 0.0, 'harm': {'harm_rate': None}}),
                ('tent', tent), ('eata', eata)]:
    h = r.get('harm', {})
    print(f"{name:7s} AUROC={r['after']['auroc']:.4f} F1={r['after']['f1']:.4f} ECE={r['after']['ece']:.4f} "
          f"cov={r.get('coverage',0):.3f} harm={h.get('harm_rate')}")
print('wrote', path)